In [18]:
import mlflow
import mlflow.spark

from pyspark.sql import SparkSession
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler 
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit


In [19]:
file_name = "work/dataset/hotel_reservations.csv"

In [20]:
spark = SparkSession.builder.appName("DecisionTreeClassification").getOrCreate()

In [21]:
data = spark.read.csv(file_name, header=True, inferSchema=True)
data.show(5)

In [22]:
data.printSchema()


In [23]:
from pyspark.sql.types import IntegerType, DoubleType, StringType
import pyspark.sql.functions as F
import pandas as pd

In [24]:
string_columns = [field.name for field in data.schema.fields if isinstance(field.dataType, StringType)]
print("String columns:", string_columns)

In [25]:
double_columns = [field.name for field in data.schema.fields if isinstance(field.dataType, DoubleType)]
print("Double columns:", double_columns)

In [26]:
integer_columns = [field.name for field in data.schema.fields if isinstance(field.dataType, IntegerType)]
print("Integer columns:", integer_columns)

In [27]:
(
    data.select([
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c) for c in data.columns
        ]
    ).show()
)

In [28]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)


In [29]:
mlflow.set_experiment("hr_decision_tree")
mlflow.pyspark.ml.autolog()
mlflow.set_tracking_uri("http://localhost:5000")

In [30]:
mlflow.end_run()

In [31]:
pipeline_stages = []
#mlflow.end_run()
with mlflow.start_run(run_name="decision_tree_classification"):

    categorical_columns = [
        'type_of_meal_plan',
        'room_type_reserved',
        'arrival_month',
        'market_segment_type'
    ]

    numerical_columns = [
    'no_of_adults',
    'no_of_children',
    'no_of_weekend_nights',
    'no_of_week_nights',
    'required_car_parking_space',
    'lead_time',
    'repeated_guest',
    'no_of_previous_cancellations',
    'no_of_previous_bookings_not_canceled',
    'avg_price_per_room',
    'no_of_special_requests'
    ]

    imputer = Imputer(inputCols=numerical_columns, outputCols=[f"{col}_imputed" for col in numerical_columns])
    pipeline_stages.append(imputer)

    for col in categorical_columns:
        indexer = StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
        #indexer_df = indexer.fit(data).transform(data)
        #indexer_df.select(col, f"{col}_indexed").show(2)

        encoder = OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded", dropLast=False)
        #encoder_df = encoder.fit(indexer_df).transform(indexer_df)
        #encoder_df.select(f'{col}', f'{col}_encoded').show(2)
        
        pipeline_stages += [indexer, encoder]


    input_columns = [col + "_encoded" for col in categorical_columns] + [col + "_imputed" for col in numerical_columns]
    print("Input columns for VectorAssembler:", input_columns)

    pipeline_stages += [StringIndexer(inputCol="booking_status", outputCol="booking_status_indexed", handleInvalid="keep")]

    assembler = VectorAssembler(inputCols=input_columns, outputCol="features_unscaled", handleInvalid="keep")
    scaler = StandardScaler(inputCol="features_unscaled", outputCol="features") 
    dt = DecisionTreeClassifier(labelCol="booking_status_indexed", featuresCol="features")
    pipeline_stages += [assembler, scaler, dt]
    pipeline = Pipeline(stages=pipeline_stages)

    grid = (
        ParamGridBuilder()
            .addGrid(dt.maxDepth, [5, 10])
            .addGrid(dt.maxBins, [32, 64])
            .build()
    )
    tvs = (
            TrainValidationSplit(estimator = pipeline,estimatorParamMaps=grid,evaluator = MulticlassClassificationEvaluator(labelCol="booking_status_indexed", metricName="accuracy"),trainRatio=0.8)
           )

    model = tvs.fit(train_data)
    mlflow.spark.log_model(model.bestModel, "decision_tree_model")
    
    # Evaluate the model on the test set
    predictions = model.transform(test_data)
    evaluator = MulticlassClassificationEvaluator(labelCol="booking_status_indexed", metricName="accuracy")
    accuracy = evaluator.evaluate(predictions)
    mlflow.log_metric("test_accuracy", accuracy)
    print(f"Test Accuracy: {accuracy:.4f}")

    # Log feature importance plot (if available)
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots()
        model.bestModel.stages[-1].featureImportances.toArray().plot(kind="barh", ax=ax)
        plt.title("Feature Importance")
        mlflow.log_figure(fig, "feature_importance.png")
    except Exception as e:
        print(f"Failed to log feature importance plot: {e}")
